In [ ]:
# @title
import os
import nibabel as nib
import torch
from torch.utils.data import Dataset
import torch.nn.functional as F
import random
import torchvision.transforms.functional as TF

class BraTSMRIDataset(Dataset):
    def __init__(self, root_dir, augment=False):
        self.root_dir = root_dir
        self.samples = []
        self.augment = augment
        self._prepare_samples()

    """
    It actually iterates through all the patients and finds the center of the tumor z
    and then the slice numbers that are to be included within the boundaries of the 3d volume
    and append all the slice numbers to the samples
    """

    def _prepare_samples(self):
        print(f"Checking root directory: {self.root_dir}")
        if not os.path.exists(self.root_dir):
            print(f"Error: Root directory does not exist: {self.root_dir}")
            return

        patient_dirs = []

        # Check if root_dir itself is a patient directory (contains a 'seg' file)
        if any("seg" in f for f in os.listdir(self.root_dir) if os.path.isfile(os.path.join(self.root_dir, f))):
            patient_dirs.append(self.root_dir)
            print(f"Treating root directory as a single patient folder: {self.root_dir}")
        else:
            # Otherwise, assume root_dir contains multiple patient subdirectories
            items_in_root = sorted(os.listdir(self.root_dir))
            for item in items_in_root:
                full_path = os.path.join(self.root_dir, item)
                if os.path.isdir(full_path):
                    patient_dirs.append(full_path)
            print(f"Found {len(patient_dirs)} patient subdirectories in {self.root_dir}")

        if not patient_dirs:
            print(f"No valid patient directories found in {self.root_dir}")
            return

        for patient_path in patient_dirs:
            try:
                seg_files = [f for f in os.listdir(patient_path) if "seg" in f]
                if not seg_files:
                    print(f"No segmentation file found for patient: {patient_path}. Skipping.")
                    continue
                seg_path = os.path.join(patient_path, seg_files[0])

                seg = nib.load(seg_path).get_fdata()
                depth = seg.shape[2]

                tumor_slices = [i for i in range(depth) if seg[:, :, i].sum() > 0]
                if len(tumor_slices) == 0:
                    print(f"No tumor slices found for patient: {patient_path}. Skipping.")
                    continue

                z_min, z_max = min(tumor_slices), max(tumor_slices)
                num_tumor = len(tumor_slices)

                for z in tumor_slices:
                    self.samples.append((patient_path, z))

                left_available  = z_min
                right_available = depth - z_max - 1

                left_to_take  = min(num_tumor, left_available)
                right_to_take = min(num_tumor, right_available)

                for i in range(1, left_to_take + 1):
                    self.samples.append((patient_path, z_min - i))

                for i in range(1, right_to_take + 1):
                    self.samples.append((patient_path, z_max + i))
                print(f"Added {len(tumor_slices) + left_to_take + right_to_take} samples for patient {os.path.basename(patient_path)}")
            except Exception as e:
                print(f"Error processing patient {patient_path}: {e}")

    def __len__(self):
        return len(self.samples)

    def _normalize_channel(self, x):
        return (x - x.min()) / (x.max() - x.min() + 1e-8)

    def _augment(self, image, mask):
        if random.random() < 0.5:
            image = TF.hflip(image)
            mask  = TF.hflip(mask)

        if random.random() < 0.5:
            image = TF.vflip(image)
            mask  = TF.vflip(mask)

        # simple spatial consistency augmentation so that the model does not get overfitted spatially
        angle = random.uniform(-10, 10)
        image = TF.rotate(image, angle, interpolation=TF.InterpolationMode.BILINEAR)
        mask  = TF.rotate(mask, angle, interpolation=TF.InterpolationMode.NEAREST)

        return image, mask

    def __getitem__(self, idx):
        patient_path, slice_idx = self.samples[idx]

        flair = nib.load(os.path.join(patient_path, [f for f in os.listdir(patient_path) if "flair" in f][0])).get_fdata()
        t1    = nib.load(os.path.join(patient_path, [f for f in os.listdir(patient_path) if "_t1." in f][0])).get_fdata()
        t1ce  = nib.load(os.path.join(patient_path, [f for f in os.listdir(patient_path) if "t1ce" in f][0])).get_fdata()
        t2    = nib.load(os.path.join(patient_path, [f for f in os.listdir(patient_path) if "_t2." in f][0])).get_fdata()
        seg   = nib.load(os.path.join(patient_path, [f for f in os.listdir(patient_path) if "seg" in f][0])).get_fdata()

        # Extract 3D patch centered at slice_idx (depth of 16 slices for proper pooling)
        depth = 16
        z_start = max(0, slice_idx - depth // 2)
        z_end = min(flair.shape[2], z_start + depth)

        # Pad if necessary
        if z_end - z_start < depth:
            z_start = max(0, z_end - depth)

        flair = torch.tensor(flair[:, :, z_start:z_end]).unsqueeze(0).float()
        t1    = torch.tensor(t1[:, :, z_start:z_end]).unsqueeze(0).float()
        t1ce  = torch.tensor(t1ce[:, :, z_start:z_end]).unsqueeze(0).float()
        t2    = torch.tensor(t2[:, :, z_start:z_end]).unsqueeze(0).float()
        seg   = torch.tensor(seg[:, :, z_start:z_end]).unsqueeze(0).float()

        # per-channel normalization
        flair = self._normalize_channel(flair)
        t1    = self._normalize_channel(t1)
        t1ce  = self._normalize_channel(t1ce)
        t2    = self._normalize_channel(t2)

        image = torch.cat([flair, t1, t1ce, t2], dim=0)
        mask = (seg > 0).float()

        # Reduce resolution to save memory
        image = F.interpolate(image.unsqueeze(0), size=(8, 64, 64),
                              mode="trilinear", align_corners=False).squeeze(0)
        mask = F.interpolate(mask.unsqueeze(0), size=(8, 64, 64),
                             mode="nearest").squeeze(0)

        if self.augment:
            image, mask = self._augment(image, mask)


        return image, mask

In [28]:
# @title
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

class U_Net(nn.Module):
    """3D U-Net for Brain Tumor Segmentation

    Architecture:
    - Encoder: 4 levels of downsampling with convolutional blocks
    - Bottleneck: Deepest level with convolutional block
    - Decoder: 4 levels of upsampling with skip connections
    - Output: Segmentation map with configurable number of classes
    """

    def __init__(self, in_channels=4, out_channels=4):
        """
        Args:
            in_channels: Number of input channels (default 4 for BraTS: T1, T1ce, T2, FLAIR)
            out_channels: Number of output classes (default 4: background, necrotic, edema, enhancing)
        """
        super().__init__()

        # Encoder (Downsampling) - Level 1
        self.enc1_conv1 = nn.Conv3d(in_channels, 64, kernel_size=3, padding=1)
        self.enc1_bn1 = nn.BatchNorm3d(64)
        self.enc1_conv2 = nn.Conv3d(64, 64, kernel_size=3, padding=1)
        self.enc1_bn2 = nn.BatchNorm3d(64)
        self.pool1 = nn.MaxPool3d(kernel_size=2, stride=2)

        # Encoder - Level 2
        self.enc2_conv1 = nn.Conv3d(64, 128, kernel_size=3, padding=1)
        self.enc2_bn1 = nn.BatchNorm3d(128)
        self.enc2_conv2 = nn.Conv3d(128, 128, kernel_size=3, padding=1)
        self.enc2_bn2 = nn.BatchNorm3d(128)
        self.pool2 = nn.MaxPool3d(kernel_size=2, stride=2)

        # Encoder - Level 3
        self.enc3_conv1 = nn.Conv3d(128, 256, kernel_size=3, padding=1)
        self.enc3_bn1 = nn.BatchNorm3d(256)
        self.enc3_conv2 = nn.Conv3d(256, 256, kernel_size=3, padding=1)
        self.enc3_bn2 = nn.BatchNorm3d(256)
        self.pool3 = nn.MaxPool3d(kernel_size=2, stride=2)

        # Encoder - Level 4
        self.enc4_conv1 = nn.Conv3d(256, 512, kernel_size=3, padding=1)
        self.enc4_bn1 = nn.BatchNorm3d(512)
        self.enc4_conv2 = nn.Conv3d(512, 512, kernel_size=3, padding=1)
        self.enc4_bn2 = nn.BatchNorm3d(512)
        self.pool4 = nn.MaxPool3d(kernel_size=2, stride=2)

        # Bottleneck
        self.bottleneck_conv1 = nn.Conv3d(512, 1024, kernel_size=3, padding=1)
        self.bottleneck_bn1 = nn.BatchNorm3d(1024)
        self.bottleneck_conv2 = nn.Conv3d(1024, 1024, kernel_size=3, padding=1)
        self.bottleneck_bn2 = nn.BatchNorm3d(1024)

        # Decoder (Upsampling) - Level 4
        self.up4 = nn.ConvTranspose3d(1024, 512, kernel_size=2, stride=2)
        self.dec4_conv1 = nn.Conv3d(1024, 512, kernel_size=3, padding=1)
        self.dec4_bn1 = nn.BatchNorm3d(512)
        self.dec4_conv2 = nn.Conv3d(512, 512, kernel_size=3, padding=1)
        self.dec4_bn2 = nn.BatchNorm3d(512)

        # Decoder - Level 3
        self.up3 = nn.ConvTranspose3d(512, 256, kernel_size=2, stride=2)
        self.dec3_conv1 = nn.Conv3d(512, 256, kernel_size=3, padding=1)
        self.dec3_bn1 = nn.BatchNorm3d(256)
        self.dec3_conv2 = nn.Conv3d(256, 256, kernel_size=3, padding=1)
        self.dec3_bn2 = nn.BatchNorm3d(256)

        # Decoder - Level 2
        self.up2 = nn.ConvTranspose3d(256, 128, kernel_size=2, stride=2)
        self.dec2_conv1 = nn.Conv3d(256, 128, kernel_size=3, padding=1)
        self.dec2_bn1 = nn.BatchNorm3d(128)
        self.dec2_conv2 = nn.Conv3d(128, 128, kernel_size=3, padding=1)
        self.dec2_bn2 = nn.BatchNorm3d(128)

        # Decoder - Level 1
        self.up1 = nn.ConvTranspose3d(128, 64, kernel_size=2, stride=2)
        self.dec1_conv1 = nn.Conv3d(128, 64, kernel_size=3, padding=1)
        self.dec1_bn1 = nn.BatchNorm3d(64)
        self.dec1_conv2 = nn.Conv3d(64, 64, kernel_size=3, padding=1)
        self.dec1_bn2 = nn.BatchNorm3d(64)

        # Output layer
        self.final_conv = nn.Conv3d(64, out_channels, kernel_size=1)

        # Activation function
        self.relu = nn.ReLU(inplace=True)

    def _conv_block(self, x, conv1, bn1, conv2, bn2):
        """Helper method to apply a convolutional block"""
        x = conv1(x)
        x = bn1(x)
        x = self.relu(x)
        x = conv2(x)
        x = bn2(x)
        x = self.relu(x)
        return x

    def forward(self, x):
        # Encoder with skip connections
        # Level 1
        enc1 = self._conv_block(x, self.enc1_conv1, self.enc1_bn1, self.enc1_conv2, self.enc1_bn2)
        x = self.pool1(enc1)

        # Level 2
        enc2 = self._conv_block(x, self.enc2_conv1, self.enc2_bn1, self.enc2_conv2, self.enc2_bn2)
        x = self.pool2(enc2)

        # Level 3
        enc3 = self._conv_block(x, self.enc3_conv1, self.enc3_bn1, self.enc3_conv2, self.enc3_bn2)
        x = self.pool3(enc3)

        # Level 4
        enc4 = self._conv_block(x, self.enc4_conv1, self.enc4_bn1, self.enc4_conv2, self.enc4_bn2)
        x = self.pool4(enc4)

        # Bottleneck
        x = self._conv_block(x, self.bottleneck_conv1, self.bottleneck_bn1, self.bottleneck_conv2, self.bottleneck_bn2)

        # Decoder with skip connections
        # Level 4
        x = self.up4(x)
        x = torch.cat([x, enc4], dim=1)
        x = self._conv_block(x, self.dec4_conv1, self.dec4_bn1, self.dec4_conv2, self.dec4_bn2)

        # Level 3
        x = self.up3(x)
        x = torch.cat([x, enc3], dim=1)
        x = self._conv_block(x, self.dec3_conv1, self.dec3_bn1, self.dec3_conv2, self.dec3_bn2)

        # Level 2
        x = self.up2(x)
        x = torch.cat([x, enc2], dim=1)
        x = self._conv_block(x, self.dec2_conv1, self.dec2_bn1, self.dec2_conv2, self.dec2_bn2)

        # Level 1
        x = self.up1(x)
        x = torch.cat([x, enc1], dim=1)
        x = self._conv_block(x, self.dec1_conv1, self.dec1_bn1, self.dec1_conv2, self.dec1_bn2)

        # Output
        x = self.final_conv(x)

        return x

In [29]:
# @title
import torch
import torch.nn as nn

class DiceLoss(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, inputs, targets, smooth=1.0):
        inputs = inputs.view(-1)
        targets = targets.view(-1)

        intersection = (inputs * targets).sum()
        dice = (2.0 * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)
        return 1.0 - dice

In [31]:
import torch
import torch.nn as nn
import torch.optim as optim


# checks wether the gpu is avaialble or not
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# loads the dataset for the training
dataset = BraTSMRIDataset("/content/brats2021", augment=True)
loader = DataLoader(dataset, batch_size=1, shuffle=True) # Further reduced batch size to 1

Checking root directory: /content/brats2021
Treating root directory as a single patient folder: /content/brats2021
Added 155 samples for patient brats2021


In [32]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

# Clear previous model and cache if they exist
if 'model' in locals() and model is not None:
    del model
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# training
model = U_Net(out_channels=1).to(device) # Move model to the selected device and set out_channels to 1
criterion = DiceLoss()
optimizer = optim.Adam(model.parameters(),lr=0.001)
epochs = 10

# Initialize variables for plotting
epoch_losses = []
batch_losses = []

for epoch in range(epochs):
    epoch_loss = 0.0
    batch_count = 0
    for i, (images, masks) in enumerate(loader):
        images = images.to(device)
        masks = masks.to(device)

        model.train()

        # forward pass
        outputs = model(images)
        loss = criterion(outputs, masks)

        # backward pass and optimization
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()
        batch_losses.append(loss.item())
        batch_count += 1

        if (i+1) % 10 == 0:
            print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(loader)}], Loss: {loss.item():.4f}")


    # Calculate average loss for epoch
    avg_epoch_loss = epoch_loss / batch_count
    epoch_losses.append(avg_epoch_loss)
    print(f"Epoch [{epoch+1}/{epochs}] - Average Loss: {avg_epoch_loss:.4f}\n")

print("="*60)
print("Training Complete!")
print("="*60 + "\n")

# Plotting
fig = plt.figure(figsize=(16, 10))

# Create a 2x2 grid for plots
gs = fig.add_gridspec(2, 2, hspace=0.3, wspace=0.3)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax3 = fig.add_subplot(gs[1, :])

# Plot 1: Loss per epoch (with trend line)
ax1.plot(epoch_losses, marker='o', linewidth=2.5, markersize=8, color='#2E86AB', label='Epoch Loss')
ax1.fill_between(range(len(epoch_losses)), epoch_losses, alpha=0.3, color='#2E86AB')
ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
ax1.set_ylabel('Dice Loss', fontsize=12, fontweight='bold')
ax1.set_title('Average Loss per Epoch', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3, linestyle='--')
ax1.set_xticks(range(len(epoch_losses)))
ax1.legend()

# Plot 2: Loss statistics
stats_text = f"""
Training Statistics:
━━━━━━━━━━━━━━━━━━━━━
Total Epochs: {epochs}
Total Batches: {len(batch_losses)}
Final Loss: {epoch_losses[-1]:.6f}
Best Loss: {min(epoch_losses):.6f}
Worst Loss: {max(epoch_losses):.6f}
Avg Loss: {sum(epoch_losses)/len(epoch_losses):.6f}

Device: {device}
Batch Size: {loader.batch_size}
Learning Rate: 0.001
Optimizer: Adam
Loss Function: Dice Loss
"""
ax2.text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
ax2.axis('off')

# Plot 3: Loss per batch (with moving average)
window_size = 10
moving_avg = []
for i in range(len(batch_losses)):
    if i < window_size:
        moving_avg.append(sum(batch_losses[:i+1]) / (i+1))
    else:
        moving_avg.append(sum(batch_losses[i-window_size+1:i+1]) / window_size)

ax3.plot(batch_losses, linewidth=0.8, color='#A23B72', alpha=0.5, label='Batch Loss')
ax3.plot(moving_avg, linewidth=2.5, color='#F18F01', label=f'Moving Average (window={window_size})')
ax3.set_xlabel('Batch', fontsize=12, fontweight='bold')
ax3.set_ylabel('Dice Loss', fontsize=12, fontweight='bold')
ax3.set_title('Loss per Batch (All Epochs)', fontsize=14, fontweight='bold')
ax3.grid(True, alpha=0.3, linestyle='--')
ax3.legend(loc='upper right')

plt.suptitle('Brain Tumor Segmentation - Training Progress', fontsize=16, fontweight='bold', y=0.995)
plt.savefig('training_loss_plot.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("Plot saved as 'training_loss_plot.png'")
print("="*60)

OutOfMemoryError: CUDA out of memory. Tried to allocate 2.00 MiB. GPU 0 has a total capacity of 14.74 GiB of which 2.12 MiB is free. Process 6257 has 14.74 GiB memory in use. Of the allocated memory 14.60 GiB is allocated by PyTorch, and 16.28 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)